# CeliGuard Celiac Risk Training Pipeline`n

In [ ]:
import os
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
HF_MAIN_DATASET = os.getenv("HF_MAIN_DATASET_SECRET")
HF_MAIN_DATASET = "ayush-yadavv/nhanes_celiac_data"
HF_TRAIN_SPLIT = "train"
HF_TEST_SPLIT = "test"
HF_MAIN_MAX_ROWS = None
# Imbalance handling configuration for main model training.
USE_CLASS_WEIGHT_BALANCING = True
USE_RANDOM_OVERSAMPLING = True
# Resolve paths correctly whether the notebook kernel starts in the repo root or train/.
CWD = Path.cwd().resolve()
if (CWD / "train").exists() and (CWD / "models").exists():
    PROJECT_ROOT = CWD
elif CWD.name == "train" and (CWD.parent / "models").exists():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD
# Keep model output compatible with backend defaults.
MODEL_OUTPUT_DIR = os.getenv("MODEL_OUTPUT_DIR")
OUTPUT_DIR = Path(MODEL_OUTPUT_DIR).expanduser() if MODEL_OUTPUT_DIR else PROJECT_ROOT / "models"
OUTPUT_MODEL_PATH = OUTPUT_DIR / "celiac_risk_model.pkl"
OUTPUT_METADATA_PATH = OUTPUT_DIR / "model_metadata.pkl"


In [ ]:
NUMERIC_FEATURES = [
    "age_at_diagnosis",
    "current_age",
    "years_of_symptoms_before_diagnosis",
    "bmi",
    "followup_years",
]

CATEGORICAL_FEATURES = [
    "sex",
    "marsh_grade_at_diagnosis",
    "mucosal_healing_on_followup",
    "rcd_type",
    "smoking_status",
    "gfd_adherence",
    "family_history_of_malignancy",
    "hla_risk",
]

TARGET_COLUMN = "celiac_serology_risk"
EXPECTED_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET_COLUMN]

VALID_CATEGORIES = {
    "sex": ["Male", "Female"],
    "marsh_grade_at_diagnosis": ["0", "1", "2", "3a", "3b", "3c"],
    "mucosal_healing_on_followup": ["Yes", "No"],
    "rcd_type": ["None", "RCD_I", "RCD_II"],
    "smoking_status": ["Never", "Former", "Current"],
    "gfd_adherence": ["Poor", "Partial", "Good", "Excellent"],
    "family_history_of_malignancy": ["Yes", "No"],
    "hla_risk": ["Low", "Medium", "High"],
}

NUMERIC_RANGES = {
    "age_at_diagnosis": (5, 80),
    "current_age": (5, 90),
    "years_of_symptoms_before_diagnosis": (0, 15),
    "bmi": (16, 35),
    "followup_years": (0, 20),
}


In [ ]:
def load_main_hf_dataset(dataset_name: str, split: str, max_rows: int | None = None) -> tuple[pd.DataFrame, str]:
    from datasets import load_dataset

    split_expr = split if max_rows is None else f"{split}[:{max_rows}]"
    df = load_dataset(dataset_name, split=split_expr).to_pandas()
    source = f"hf:{dataset_name}/{split_expr}"
    return df, source

In [ ]:
def clean_and_validate(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    cleaned = df.copy()
    report = {"rows_before": len(cleaned)}

    missing_feature_cols = [c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES if c not in cleaned.columns]
    if missing_feature_cols:
        raise ValueError(f"Missing required feature columns: {missing_feature_cols}")

    for col in NUMERIC_FEATURES:
        low, high = NUMERIC_RANGES[col]
        cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")
        median_value = cleaned[col].median()
        if pd.isna(median_value):
            median_value = (low + high) / 2
        cleaned[col] = cleaned[col].fillna(median_value).clip(lower=low, upper=high)

    for col in CATEGORICAL_FEATURES:
        allowed = VALID_CATEGORIES[col]
        cleaned[col] = cleaned[col].astype(str).str.strip()
        mode_value = cleaned[col].mode(dropna=True)
        fallback = mode_value.iloc[0] if not mode_value.empty and mode_value.iloc[0] in allowed else allowed[0]
        cleaned[col] = cleaned[col].where(cleaned[col].isin(allowed), fallback)

    if TARGET_COLUMN not in cleaned.columns:
        raise ValueError(
            f"Missing required target column {TARGET_COLUMN!r}. "
            "Expected target labels from the Hugging Face dataset."
        )

    cleaned[TARGET_COLUMN] = pd.to_numeric(cleaned[TARGET_COLUMN], errors="coerce")
    cleaned[TARGET_COLUMN] = cleaned[TARGET_COLUMN].fillna(1).astype(int).clip(lower=0, upper=2)
    report["target_mode"] = "from_data"

    cleaned = cleaned.dropna(subset=NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET_COLUMN]).reset_index(drop=True)

    report["rows_after"] = len(cleaned)
    report["dropped_rows"] = report["rows_before"] - report["rows_after"]
    report["class_distribution"] = cleaned[TARGET_COLUMN].value_counts(normalize=True).sort_index().to_dict()

    return cleaned, report


In [ ]:
raw_train_df, train_source = load_main_hf_dataset(
    dataset_name=HF_MAIN_DATASET,
    split=HF_TRAIN_SPLIT,
    max_rows=HF_MAIN_MAX_ROWS,
)
raw_test_df, test_source = load_main_hf_dataset(
    dataset_name=HF_MAIN_DATASET,
    split=HF_TEST_SPLIT,
    max_rows=None,
)

clean_train_df, train_cleaning_report = clean_and_validate(raw_train_df)
clean_test_df, test_cleaning_report = clean_and_validate(raw_test_df)
data_source = f"hf:{HF_MAIN_DATASET}/{HF_TRAIN_SPLIT}+{HF_TEST_SPLIT}"

print(f"Train source: {train_source}")
print(f"Train rows: {train_cleaning_report['rows_before']} -> {train_cleaning_report['rows_after']}")
print("Train class distribution (normalized):")
print(pd.Series(train_cleaning_report['class_distribution']).sort_index())
print()
print(f"Test source: {test_source}")
print(f"Test rows: {test_cleaning_report['rows_before']} -> {test_cleaning_report['rows_after']}")
print("Test class distribution (normalized):")
print(pd.Series(test_cleaning_report['class_distribution']).sort_index())

clean_train_df.head()

In [ ]:
X_train = clean_train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_train = clean_train_df[TARGET_COLUMN]

X_test = clean_test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_test = clean_test_df[TARGET_COLUMN]

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

class_weight_option = "balanced" if USE_CLASS_WEIGHT_BALANCING else None

lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                class_weight=class_weight_option,
            ),
        ),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                class_weight=class_weight_option,
            ),
        ),
    ]
)


In [ ]:
def oversample_training_data(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    random_state: int,
) -> tuple[pd.DataFrame, pd.Series]:
    class_counts = y_train.value_counts()
    if class_counts.empty:
        return X_train, y_train

    max_count = int(class_counts.max())
    sampled_indices = []

    for cls, count in class_counts.items():
        cls_index = y_train[y_train == cls].index
        if count < max_count:
            sampled = np.random.default_rng(random_state).choice(cls_index.to_numpy(), size=max_count, replace=True)
            sampled_indices.extend(sampled.tolist())
        else:
            sampled_indices.extend(cls_index.tolist())

    sampled_indices = pd.Index(sampled_indices)
    X_bal = X_train.loc[sampled_indices].copy()
    y_bal = y_train.loc[sampled_indices].copy()

    # Shuffle after oversampling to avoid class blocks.
    shuffled = np.random.default_rng(random_state).permutation(len(X_bal))
    X_bal = X_bal.iloc[shuffled].reset_index(drop=True)
    y_bal = y_bal.iloc[shuffled].reset_index(drop=True)

    return X_bal, y_bal


def evaluate_model(name: str, pipeline: Pipeline, X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: pd.Series, y_test: pd.Series) -> dict:
    X_fit, y_fit = X_train, y_train
    if USE_RANDOM_OVERSAMPLING:
        X_fit, y_fit = oversample_training_data(X_train, y_train, random_state=RANDOM_STATE)

    pipeline.fit(X_fit, y_fit)
    y_pred = pipeline.predict(X_test)

    all_labels = sorted(np.unique(np.concatenate([y_test.values, y_pred])))
    label_names = {0: "Low Celiac Risk", 1: "Moderate Celiac Risk", 2: "High Celiac Risk"}
    target_names = [label_names.get(label, f"Class {label}") for label in all_labels]

    report_dict = classification_report(
        y_test,
        y_pred,
        labels=all_labels,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )

    metrics = {
        "name": name,
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "macro_f1": float(f1_score(y_test, y_pred, labels=all_labels, average="macro", zero_division=0)),
        "classification_report": classification_report(
            y_test,
            y_pred,
            labels=all_labels,
            target_names=target_names,
            output_dict=False,
            zero_division=0,
        ),
        "classification_report_dict": report_dict,
        "confusion_matrix": confusion_matrix(y_test, y_pred, labels=all_labels),
        "labels": all_labels,
        "y_pred": y_pred,
        "train_class_distribution": y_train.value_counts(normalize=True).sort_index().to_dict(),
        "fit_class_distribution": y_fit.value_counts(normalize=True).sort_index().to_dict(),
        "pipeline": pipeline,
    }
    return metrics

lr_metrics = evaluate_model("Logistic Regression", lr_pipeline, X_train, X_test, y_train, y_test)
rf_metrics = evaluate_model("Random Forest", rf_pipeline, X_train, X_test, y_train, y_test)

for metrics in [lr_metrics, rf_metrics]:
    print("=" * 80)
    print(metrics['name'])
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Balanced accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print("Labels:", metrics['labels'])
    print("Train class distribution:", metrics['train_class_distribution'])
    print("Fit class distribution:", metrics['fit_class_distribution'])
    print("Classification report:")
    print(metrics['classification_report'])
    print("Confusion matrix:")
    print(metrics['confusion_matrix'])


In [ ]:
label_names = {0: "Low Celiac Risk", 1: "Moderate Celiac Risk", 2: "High Celiac Risk"}

train_dist = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_dist = pd.Series(y_test).value_counts(normalize=True).sort_index()
fit_lr_dist = pd.Series(lr_metrics["fit_class_distribution"]).sort_index()
fit_rf_dist = pd.Series(rf_metrics["fit_class_distribution"]).sort_index()

dist_df = pd.DataFrame(
    {
        "train": train_dist,
        "test": test_dist,
        "fit_lr": fit_lr_dist,
        "fit_rf": fit_rf_dist,
    }
).fillna(0.0)
dist_df.index = [label_names.get(int(i), str(i)) for i in dist_df.index]

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Class distribution overview
ax = axes[0, 0]
dist_df.plot(kind="bar", ax=ax)
ax.set_title("Class Distribution (Original vs Fitted Training)")
ax.set_ylabel("Proportion")
ax.set_xlabel("Class")
ax.grid(axis="y", alpha=0.3)

# Confusion matrix: logistic regression
ax = axes[0, 1]
lr_labels = [label_names.get(int(i), str(i)) for i in lr_metrics["labels"]]
sns.heatmap(
    lr_metrics["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=lr_labels,
    yticklabels=lr_labels,
    ax=ax,
)
ax.set_title("Logistic Regression Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

# Confusion matrix: random forest
ax = axes[1, 0]
rf_labels = [label_names.get(int(i), str(i)) for i in rf_metrics["labels"]]
sns.heatmap(
    rf_metrics["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=rf_labels,
    yticklabels=rf_labels,
    ax=ax,
)
ax.set_title("Random Forest Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

# Per-class recall comparison
ax = axes[1, 1]
lr_recall = {}
for label in lr_metrics["labels"]:
    name = label_names.get(int(label), str(label))
    lr_recall[name] = lr_metrics["classification_report_dict"].get(name, {}).get("recall", 0.0)
rf_recall = {}
for label in rf_metrics["labels"]:
    name = label_names.get(int(label), str(label))
    rf_recall[name] = rf_metrics["classification_report_dict"].get(name, {}).get("recall", 0.0)

recall_df = pd.DataFrame({"Logistic Regression": pd.Series(lr_recall), "Random Forest": pd.Series(rf_recall)}).fillna(0.0)
recall_df.plot(kind="bar", ax=ax)
ax.set_title("Per-class Recall")
ax.set_ylabel("Recall")
ax.set_xlabel("Class")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
best = max(
    [lr_metrics, rf_metrics],
    key=lambda x: (x['balanced_accuracy'], x['macro_f1'], x['accuracy']),
)
best_model = best['pipeline']

print(f"Selected model: {best['name']}")
print(f"Selected balanced accuracy: {best['balanced_accuracy']:.4f}")
print(f"Selected macro F1: {best['macro_f1']:.4f}")
print(f"Selected accuracy: {best['accuracy']:.4f}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "model_type": best['name'],
    "accuracy": best['accuracy'],
    "balanced_accuracy": best['balanced_accuracy'],
    "macro_f1": best['macro_f1'],
    "data_source": data_source,
    "target_mode": train_cleaning_report['target_mode'],
    "target_column": TARGET_COLUMN,
    "target_type": "celiac_risk",
    "not_malignancy_model": True,
    "intended_use": "Research prototype for estimating celiac risk from the configured Hugging Face dataset.",
    "limitations": "Research prototype only; not for clinical diagnosis or treatment decisions.",
    "effective_feature_note": "Feature coverage and quality depend on the configured Hugging Face dataset schema.",
    "class_weight_balancing": USE_CLASS_WEIGHT_BALANCING,
    "random_oversampling": USE_RANDOM_OVERSAMPLING,
}

joblib.dump(best_model, OUTPUT_MODEL_PATH)
joblib.dump(metadata, OUTPUT_METADATA_PATH)

print(f"Saved model: {OUTPUT_MODEL_PATH.resolve()}")
print(f"Saved metadata: {OUTPUT_METADATA_PATH.resolve()}")


In [ ]:
sample = X_test.iloc[[0]].copy()
sample_truth = int(y_test.iloc[0])
sample_pred = int(best_model.predict(sample)[0])
sample_proba = best_model.predict_proba(sample)[0]

risk_labels = {0: "Low Celiac Risk", 1: "Moderate Celiac Risk", 2: "High Celiac Risk"}

print("Smoke test prediction:")
print(f"Ground truth: {sample_truth} ({risk_labels[sample_truth]})")
print(f"Predicted: {sample_pred} ({risk_labels[sample_pred]})")
print(f"Probabilities [Low, Moderate, High]: {sample_proba}")

In [ ]:
if hasattr(best_model, "predict_proba"):
    proba = best_model.predict_proba(X_test)
    class_to_idx = {int(c): idx for idx, c in enumerate(best_model.classes_)}

    classes_to_plot = [cls for cls in [1, 2] if cls in class_to_idx]
    if not classes_to_plot:
        classes_to_plot = [int(best_model.classes_[0])]

    fig, axes = plt.subplots(1, len(classes_to_plot), figsize=(7 * len(classes_to_plot), 4))
    if len(classes_to_plot) == 1:
        axes = [axes]

    for ax, cls in zip(axes, classes_to_plot):
        idx = class_to_idx[cls]
        cls_name = label_names.get(int(cls), f"Class {cls}")
        all_scores = proba[:, idx]
        positive_scores = all_scores[y_test.values == cls]

        sns.histplot(all_scores, bins=30, stat="density", alpha=0.35, label="All samples", ax=ax)
        if len(positive_scores) > 0:
            sns.histplot(positive_scores, bins=30, stat="density", alpha=0.55, label=f"True {cls_name}", ax=ax)
        ax.set_title(f"Predicted Probability Distribution: {cls_name}")
        ax.set_xlabel("Predicted probability")
        ax.set_ylabel("Density")
        ax.legend()

    plt.tight_layout()
    plt.show()
else:
    print("Best model has no predict_proba; skipping probability plots")
